----
# **<span style="color:DarkSlateBlue">PROYECTO BOOKING_REVIEWS</span>**
# **<span style="color:DarkSlateBlue">Limpieza de datos</span>**
----

---
---
## <span style="color:gray">**1. Importación de librerías**</span> 📂

In [113]:
# Tratamiento de datos
import numpy as np
import pandas as pd 
from IPython.display import display
import pycountry

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de ruta
import sys
sys.path.append('../')

# Importación de funciones personalizadas
from src.soporte2_limpieza import *


---
---
## <span style="color:gray">**2. Carga de datos**</span> 📥

In [114]:
booking_raw = pd.read_csv("../data/raw/hotel_booking.csv")

reviews_raw = pd.read_csv("../data/raw/hotel_reviews.csv")

In [115]:
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

---
---

----
# <span style="color:DarkSlateBlue">**Desarrollo del proyecto - 2**</span>
----


---
---
## <span style="color:gray">**Limpieza de datos**</span> 🧹

Antes de continuar con el análisis exploratorio de datos (EDA), creamos copias de los DataFrames originales.
Esto nos permite manipular, limpiar o transformar los datos sin afectar los originales, asegurando que siempre podamos recuperar la información original si es necesario.

In [116]:
booking_limpio = booking_raw.copy()
reviews_limpio = reviews_raw.copy()

---
## `booking_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>


La única columna que habría que estandarizar es `phone-number`, ya que utiliza como separador "-", en lugar de "_" como el resto de nombres de columnas. Sin embargo, al ser irrelevante para nuestro análisis, la eliminaremos en el próximo paso.

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `arrival_date_week_number` y `arrival_date_day_of_month`

Solo necesitamos el mes y año de llegada para poder unir este dataset con el de reviews. Por lo tanto, estas columnas se pueden eliminar.

- `agent`y `company`

Representan los ID de las agencias y compañías. No aportan información relevante para nuestro análisis, por lo que se eliminan.

- `days_in_waiting_list`

La mayoría de los huéspedes tienen 0 días en espera y no es una información útil para el análisis que queremos realizar. En consecuncia, la eliminaremos.

- `required_car_parking_spaces` y `total_of_special_requests`

El valor promedio es muy bajo, la desviación estándar es baja y los percentiles muestran que la mayoría de los huéspedes no requiere estas opciones. Estas columnas aportan poca información útil y se eliminan.

- `name`, `email`, `phone-number`, `credit_card` 

Nos dan información sensible del huésped que no es que no aporta valor al análisis. Se eliminan.

**Eliminamos las columnas anteriores**

In [117]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
booking_limpio = eliminar_columns(
    booking_limpio,
    ["arrival_date_week_number", "arrival_date_day_of_month", 
     "agent", "company", "days_in_waiting_list", 
     "required_car_parking_spaces", "total_of_special_requests",
     "name", "email", "phone-number", "credit_card"]
)

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos la abreviatura de los países por el nombre completo del país.**

In [118]:
def code_to_name(code):
    try:
        return pycountry.countries.get(alpha_3=code.upper()).name
    except:
        return None

booking_limpio['country'] = booking_limpio['country'].apply(code_to_name)

- **Comprobamos los datos únicos de las variables categóricas y hacemos la limpieza de algunos de los datos.**

In [119]:
col_cat = booking_limpio.select_dtypes(include=['category', 'str']).columns
datos_unicos(booking_limpio, col_cat)



Los datos únicos de la varible hotel son:

 <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible arrival_date_month son:

 <StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible meal son:

 <StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible country son:

 <StringArray>
[                        'Portugal',                   'United Kingdom',
                    'United States',                            'Spain',
                          'Ireland',                           'France',
                             

- Todas las variables presentan mayúsculas y minúsculas, lo estandarizamos todo a minúsculas.
- Las variables `hotel`, `deposit_type` y `market_segment` tienen algunos datos con espacio entre medias, lo cambiamos por "_".
- `market_segment` y `distribution_channel` tienen el caracter "/", el cual sustituimos por "_".
- `customer_type` y `reservation_status` tienen como separador de palabras "-", lo cambiamos por "_".


In [120]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
minus(booking_limpio, col_cat)

# Llamamos a la función reemplazar guardada en el archivo de soporte 
reemplazar(booking_limpio, ["hotel", "country", "deposit_type", "market_segment"], " ", "_")
reemplazar(booking_limpio, ["market_segment", "distribution_channel"], "/", "_")
reemplazar(booking_limpio, ["customer_type", "reservation_status"], "-", "_")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Cambiamos las variables `is_canceled` y `is_repeated_guest` a variables categóricas:** 

   - 1 = 'si'
   - 0 = 'no'

In [121]:
columnas = ["is_canceled", "is_repeated_guest"]

for columna in columnas:
    booking_limpio[columna] = booking_limpio[columna].replace({1:'si', 0:'no'})

- **Variable `children`**

Para convertir esta variable a números enteros, primero hay que pasar de string a float, hacer un redondeo al entero más cercano y finalmente se cambia a int:

In [122]:
booking_limpio['children'] = booking_limpio['children'].astype('Int64')

- **Variable `reservation_status_date`**

In [123]:
booking_limpio["reservation_status_date"] = (
    pd.to_datetime(booking_limpio["reservation_status_date"], 
                   format="%Y-%M-%d", 
                   errors="coerce")
                   )

Una vez convertida la variable a formato fecha, se crearán dos nuevas variables: una que contenga el año y otra el mes, con el objetivo de obtener información temporal más relevante para el análisis.

In [124]:
booking_limpio['reservation_status_year'] = booking_limpio['reservation_status_date'].dt.year

In [125]:
booking_limpio['reservation_status_month'] = booking_limpio['reservation_status_date'].dt.month_name()

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

Durante la exploración inicial del Dataset, identificamos columnas con valores nulos; a continuación, evaluaremos cómo tratarlos individualmente.

Antes de tratar los nulos generamos un DataFrame que contenga solo las columnas que tienen nulos:

In [126]:
df_nulos = booking_limpio.loc[:, booking_limpio.isnull().sum() > 0]
df_nulos.columns

Index(['children', 'country'], dtype='str')

Recordamos el porcentaje de nulos de cada columna:

In [127]:
(df_nulos.isnull().sum() / (df_nulos.shape[0])*100).round(3)

children    0.003
country     1.483
dtype: float64

- Variable `children`

Lo imputamos con la moda.

In [128]:
booking_limpio["children"] = booking_limpio["children"].fillna(booking_limpio["children"].mode()[0])


- Variable `country`

Sustituímos los valores nulos por 'unknown'.

In [129]:
booking_limpio["country"] = booking_limpio["country"].fillna('unknown') 

---
## `reviews_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>

- Estandarización de nombres de las columnas

In [130]:
estandar_columns(reviews_limpio)

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `review_total_negative_word_counts` y `review_total_positive_word_counts` 
- `lat` y `lng`

**Eliminamos las columnas anteriores**

In [131]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
reviews_limpio = eliminar_columns(
    reviews_limpio, 
    ["review_total_negative_word_counts", 
     "review_total_positive_word_counts", 
     "lat", "lng"]
     )

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos los datos de las variables categóricas a minúscula**

In [132]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
col_cat = reviews_limpio.select_dtypes(include=['category', 'str']).columns
minus(reviews_limpio, col_cat)

- Cambiamos los espacios de las columnas `hotel_address`, `hotel_name` y `reviewer_nationality` por "_".


In [133]:
reemplazar(reviews_limpio, ["hotel_address", "hotel_name", "reviewer_nationality"], " ", "_")

- Variable `tags`

In [134]:
# Cambiamos los string a lista de strings
import ast

reviews_limpio['tags'] = reviews_limpio['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [135]:
def procesar_tags(lista): 
    
    result = { 'trip_type': None, 
              'group_type': None, 
              'room_type': None, 
              'nights': None, 
              'device': None } 
    
    for item in lista: 
        if ('leisure trip' in item) or ('business trip' in item):
            result['trip_type'] = item
        elif 'night' in item:
            result['nights'] = item
        elif 'submitted' in item:
            result['device'] = item
        elif ('couple' in item) or ('family with' in item) or ('traveler' in item) or ('group' in item) or ('with friends' in item):
            result['group_type'] = item
        else:
            result['room_type'] = item
    return pd.Series(result)

In [136]:
reviews_limpio = reviews_limpio.join(reviews_limpio['tags'].apply(procesar_tags))

In [143]:
reviews_limpio = eliminar_columns(reviews_limpio, ["tags"])

- Eliminar *day* de la columna `days_since_review`

In [144]:
reemplazar(reviews_limpio, ["days_since_review"], " day", "")

### <span style="color:darkgray">**6. Eliminación de duplicados**</span>

In [145]:
reviews_limpio = reviews_limpio.drop_duplicates()

---
---
## <span style="color:gray">**Validación final de los datasets**</span> ✅

- **Comprobamos que la limpieza de ambos dataset se ha realizado correctamente**


### **`booking_limpio`** 

In [146]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(booking_limpio)



Las 3 primeras filas del Dataframe son:



,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,customer_type,adr,reservation_status,reservation_status_date,reservation_status_year,reservation_status_month
0,resort_hotel,no,342,2015,july,0,0,2,0,0,bb,portugal,direct,direct,no,0,0,c,c,3,no_deposit,transient,0.0,check_out,2015-01-01 00:07:00,2015,January
1,resort_hotel,no,737,2015,july,0,0,2,0,0,bb,portugal,direct,direct,no,0,0,c,c,4,no_deposit,transient,0.0,check_out,2015-01-01 00:07:00,2015,January
2,resort_hotel,no,7,2015,july,0,1,1,0,0,bb,united_kingdom,direct,direct,no,0,0,a,c,0,no_deposit,transient,75.0,check_out,2015-01-02 00:07:00,2015,January



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  str           
 1   is_canceled                     119390 non-null  object        
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  str           
 5   stays_in_weekend_nights         119390 non-null  int64         
 6   stays_in_week_nights            119390 non-null  int64         
 7   adults                          119390 non-null  int64         
 8   children                        119390 non-null  Int64         
 9   babies              

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
customer_type                     0
adr                               0
reservation_status                0
reservation_status_date           0
reservation_status_year           0
reservation_status_month          0
dtype: int64

Y los datos de la variable `reservation_status_month` empiezan por mayúsculas, los cambiamos a minúsculas para mantener la consistencia en todo el DataFrame:

In [147]:
minus(booking_limpio, ['reservation_status_month'])

### **`reviews_limpio`** 

In [148]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(reviews_limpio)



Las 3 primeras filas del Dataframe son:



,hotel_address,additional_number_of_scoring,review_date,average_score,hotel_name,reviewer_nationality,negative_review,total_number_of_reviews,positive_review,total_number_of_reviews_reviewer_has_given,reviewer_score,days_since_review,trip_type,group_type,room_type,nights,device
0,s_gravesandestraat_55_oost_1092_aa_amsterdam_n...,194,8/3/2017,7.7,hotel_arena,russia,i am so angry that i made this post available ...,1403,only the park outside of the hotel was beautiful,7,2.9,0s,leisure trip,couple,duplex double room,stayed 6 nights,NaN
1,s_gravesandestraat_55_oost_1092_aa_amsterdam_n...,194,8/3/2017,7.7,hotel_arena,ireland,no negative,1403,no real complaints the hotel was great great l...,7,7.5,0s,leisure trip,couple,duplex double room,stayed 4 nights,NaN
2,s_gravesandestraat_55_oost_1092_aa_amsterdam_n...,194,7/31/2017,7.7,hotel_arena,australia,rooms are nice but for elderly a bit difficult...,1403,location was good and staff were ok it is cute...,9,7.1,3s,leisure trip,family with young children,duplex double room,stayed 3 nights,submitted from a mobile device



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
Index: 515210 entries, 0 to 515737
Data columns (total 17 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   hotel_address                               515210 non-null  str    
 1   additional_number_of_scoring                515210 non-null  int64  
 2   review_date                                 515210 non-null  str    
 3   average_score                               515210 non-null  float64
 4   hotel_name                                  515210 non-null  str    
 5   reviewer_nationality                        515210 non-null  str    
 6   negative_review                             515210 non-null  str    
 7   total_number_of_reviews                     515210 non-null  int64  
 8   positive_review                             

hotel_address                                      0
additional_number_of_scoring                       0
review_date                                        0
average_score                                      0
hotel_name                                         0
reviewer_nationality                               0
negative_review                                    0
total_number_of_reviews                            0
positive_review                                    0
total_number_of_reviews_reviewer_has_given         0
reviewer_score                                     0
days_since_review                                  0
trip_type                                      15004
group_type                                         0
room_type                                        214
nights                                           192
device                                        207856
dtype: int64

Hemos verificado que la limpieza de ambos dataframes se ha realizado correctamente: 

- Los nombres de las columnas son claros y descriptivos.
- Los datos son consistentes entre sí.
- No se observan valores nulos.
- Las variables presentan formatos adecuados para análisis y modelado posteriores .

Además, las transformaciones aplicadas (como estandarización de texto, conversión de tipos numéricos y codificación de variables categóricas) aseguran que ambos dataframes puedan combinarse y utilizarse de manera confiable.

- **Guardamos los dataset limpios**

In [ ]:
booking_limpio.to_csv("../data/output/booking_limpio.csv", index=False)
reviews_limpio.to_csv("../data/output/reviews_limpio.csv", index=False)

---